# Day 30 — Practice Questions
### A/B Test Design & Analysis · Module 2 · Topic 2.19

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- **Did I size it before I ran it?**
- **Did I check the split?**
- **Does the decision come from the interval, or from the p-value?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

def n_per_arm(p0, mde, power=0.80, alpha=0.05):
    p1   = p0 + mde
    pbar = (p0 + p1) / 2
    za, zb = stats.norm.ppf(1 - alpha/2), stats.norm.ppf(power)
    return int(np.ceil((za*np.sqrt(2*pbar*(1-pbar)) +
                        zb*np.sqrt(p0*(1-p0) + p1*(1-p1)))**2 / mde**2))

def two_prop(c_a, n_a, c_b, n_b, alpha=0.05):
    p_a, p_b = c_a/n_a, c_b/n_b
    diff = p_b - p_a
    pbar = (c_a + c_b)/(n_a + n_b)
    z = diff / np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
    p = 2*(1 - stats.norm.cdf(abs(z)))
    se = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    c  = stats.norm.ppf(1 - alpha/2)
    return diff, p, (diff - c*se, diff + c*se)

print("ready")

---
# Part A — Predict the output

Eight short cells. Write your prediction first, then run it.

### A1 · How much does a smaller baseline cost you?

In [ ]:
print(n_per_arm(0.05, 0.01))      # 5% baseline, catch +1pp
print(n_per_arm(0.05, 0.005))     # 5% baseline, catch +0.5pp
print(n_per_arm(0.20, 0.01))      # 20% baseline, catch +1pp

### A2 · The test and the interval disagree about which SE to use

In [ ]:
c_a, n_a, c_b, n_b = 250, 5000, 900, 5000
p_a, p_b = c_a/n_a, c_b/n_b
pbar = (c_a + c_b)/(n_a + n_b)

pooled   = np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
unpooled = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
print(f"pooled   {pooled:.6f}")
print(f"unpooled {unpooled:.6f}")
print(f"ratio    {pooled/unpooled:.3f}")

### A3 · Significant?

In [ ]:
diff, p, (lo, hi) = two_prop(400, 10_000, 450, 10_000)
print(f"lift {diff*100:+.3f}pp   p = {p:.5f}")
print(f"95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
print(f"relative {diff/0.04:+.1%}")

### A4 · Is this split acceptable?

In [ ]:
res = stats.chisquare([20_300, 19_700], [20_000, 20_000])
print(f"split {20300/40000:.3%} / {19700/40000:.3%}")
print(f"chi2 = {res.statistic:.2f}   p = {res.pvalue:.4f}")

### A5 · How many looks before you are fooling yourself?

In [ ]:
for k in [1, 3, 5, 10]:
    print(f"{k:>2} independent tests -> {1 - 0.95**k:.1%} chance of at least one 'win'")

### A6 · Reading an interval

In [ ]:
diff, p, (lo, hi) = two_prop(1000, 10_000, 1100, 10_000)
print(f"absolute {diff*100:+.3f}pp   95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
print(f"relative {diff/0.10:+.1%}     95% CI [{lo/0.10:+.1%}, {hi/0.10:+.1%}]")
print(f"p = {p:.5f}")

### A7 · The cost of precision

In [ ]:
print(n_per_arm(0.10, 0.02))
print(n_per_arm(0.10, 0.04))
print(round(n_per_arm(0.10, 0.02) / n_per_arm(0.10, 0.04), 2))

### A8 · What power did the test actually have?

In [ ]:
def power_of(p0, p1, n, alpha=0.05):
    pbar = (p0 + p1)/2
    za = stats.norm.ppf(1 - alpha/2)
    se0 = np.sqrt(2*pbar*(1-pbar)/n)
    se1 = np.sqrt((p0*(1-p0) + p1*(1-p1))/n)
    return 1 - stats.norm.cdf((za*se0 - abs(p1-p0))/se1)

print(f"n =    900/arm -> {power_of(0.10, 0.11, 900):.1%}")
print(f"n = 14,751/arm -> {power_of(0.10, 0.11, 14751):.1%}")

---
# Part B — Spot & fix the bug

Each cell runs and produces a confident, wrong conclusion.

### B1 · "We hit significance on day 3, so we shipped early"

Both arms convert at exactly 10%. There is no effect.

In [ ]:
rng = np.random.default_rng(4008)
c_a = c_b = tot = 0
for day in range(1, 15):
    c_a += rng.binomial(1000, 0.10)
    c_b += rng.binomial(1000, 0.10)
    tot += 1000
    diff, p, ci = two_prop(c_a, tot, c_b, tot)
    if p < 0.05:
        print(f"Day {day}: lift {diff*100:+.3f}pp, p = {p:.4f} -> SIGNIFICANT, ship it!")
        break

**What is wrong, and what would the full test have shown?**

### B2 · "Randomising by pageview gives us more data, so more power"

500 users, randomised by **pageview**. Both arms are identical — there is no effect.

In [ ]:
rng = np.random.default_rng(4002)
n_users  = 500
views    = rng.poisson(8, n_users) + 1
user_eff = rng.normal(0, 1.2, n_users)          # users differ a lot from each other
arm      = np.arange(n_users) % 2
rate     = 1/(1 + np.exp(-(-2.0 + user_eff)))   # SAME for both arms: no effect
clicks   = rng.binomial(views, rate)

df = pd.DataFrame({"arm": arm, "views": views, "clicks": clicks})
g = df.groupby("arm")[["views", "clicks"]].sum()
diff, p, ci = two_prop(g.clicks[0], g.views[0], g.clicks[1], g.views[1])
print(f"pageview-level: {g.clicks[0]/g.views[0]:.4f} vs {g.clicks[1]/g.views[1]:.4f}")
print(f"lift {diff*100:+.3f}pp, p = {p:.4f}  -> significant!")

**Why is this p-value wrong?**

### B3 · "It was flat overall, but it worked brilliantly on desktop"

No effect in any segment — all eight are generated at 8% in both arms.

In [ ]:
rng = np.random.default_rng(4003)
segments = ["mobile", "desktop", "tablet", "new", "returning", "US", "EU", "APAC"]
rows = []
for s in segments:
    n = 1500
    a = rng.binomial(n, 0.08); b = rng.binomial(n, 0.08)
    diff, p, ci = two_prop(a, n, b, n)
    rows.append({"segment": s, "lift_pp": round(diff*100, 3), "p": round(p, 4)})

res = pd.DataFrame(rows).sort_values("p")
print(res.to_string(index=False))
print("\nDesktop is significant (p < 0.05). Recommend shipping to desktop users.")

**What is wrong, and what is the correct threshold?**

### B4 · "No significant difference, so the new design doesn't work"

In [ ]:
rng = np.random.default_rng(4004)
n = 900
a = rng.binomial(n, 0.10); b = rng.binomial(n, 0.10)
diff, p, ci = two_prop(a, n, b, n)
print(f"control {a}/{n}, treatment {b}/{n}")
print(f"lift {diff*100:+.3f}pp, p = {p:.4f}")
print("Conclusion: the new design has no effect. Do not ship.")

**What is missing, and what should the conclusion be?**

---
# Part C — Write the code

Five tasks. Write the function, then run the test cell underneath it.

### C1 · A test plan calculator

Your checkout page converts at **4%**. You get **5,000 users per day per arm**.

Write `plan(p0, mde, per_day)` that prints the sample size, the number of days, and the
number of days **rounded up to whole weeks**. Then use it to decide whether chasing a +0.5pp
lift is realistic.

In [ ]:
def plan(p0, mde, per_day):
    # your code here
    pass

### C2 · The smallest effect your traffic can see

The inverse question. Write `mde_for_n(p0, n)` that finds the smallest lift detectable at
80% power with `n` users per arm. Use a bisection search over `n_per_arm`.

In [ ]:
def mde_for_n(p0, n, power=0.80, alpha=0.05):
    # your code here
    pass

### C3 · A guardrail checker

Write `guardrails(df)` that takes a frame with one row per metric (`metric`, `c_a`, `n_a`,
`c_b`, `n_b`) and flags any metric that got **significantly worse**, with a Bonferroni
correction for the number of guardrails.

In [ ]:
guard = pd.DataFrame([
    # up_is_good says which direction counts as an improvement for each metric
    {"metric": "checkout_conversion", "c_a": 1483, "n_a": 14900, "c_b": 1712, "n_b": 14900, "up_is_good": True},
    {"metric": "signup_rate",         "c_a": 2100, "n_a": 14900, "c_b": 2060, "n_b": 14900, "up_is_good": True},
    {"metric": "support_ticket_rate", "c_a":  180, "n_a": 14900, "c_b":  265, "n_b": 14900, "up_is_good": False},
    {"metric": "newsletter_optin",    "c_a":  900, "n_a": 14900, "c_b":  880, "n_b": 14900, "up_is_good": True},
])

def guardrails(df, alpha=0.05):
    # your code here
    pass

### C4 · A full pre-flight check

Write `preflight(c_a, n_a, c_b, n_b, planned_n, p0, mde)` that refuses to give a verdict
unless the test passes three checks: the split is clean, the planned sample size was
reached, and the test had at least 80% power for its stated MDE.

In [ ]:
def preflight(c_a, n_a, c_b, n_b, planned_n, p0, mde):
    # your code here
    pass

### C5 · Show the winner's curse yourself

Simulate a true **+2pp** lift on a 20% baseline. For n = 500, 2,000 and the properly powered
n, report the power and the average measured lift **among the tests that won**.

In [ ]:
# your code here

---
# Part D — Challenge

Two open tasks.

### D1 · Design a test from a brief

> *"Our email signup form converts at **6%**. The design team has rebuilt it and thinks it
> will do better. We get about **3,000 visitors a day** to that page. Marketing says anything
> below a **1 percentage point** lift is not worth the migration work. Can we run this?"*

Produce a complete one-page test plan. It must state:

1. the unit of randomisation, and why
2. the primary metric, and two guardrails
3. the MDE, with the reasoning
4. n per arm, total users, and the number of days — rounded to whole weeks
5. the fixed end date rule and what would make you stop early
6. the exact decision rule, written before any data exists

Then compute the sizing, and say whether you would recommend running it.

In [ ]:
# your code here

### D2 · Audit a real-looking write-up

Here is an experiment summary of the kind you will be handed in a meeting. **There are four
separate things wrong with it.** Find all four, say what each one does to the conclusion, and
write the honest version.

> **Test: new product recommendation carousel**
>
> *We ran the new carousel against the current page for 5 days (Mon–Fri), randomising each
> pageview. 48,200 pageviews saw the control and 51,800 saw the treatment.*
>
> *Revenue per session rose from \$4.10 to \$4.38, a 6.8% increase (p = 0.041). We checked
> conversion rate, add-to-cart rate, bounce rate, session length, return visits and
> newsletter signups, and add-to-cart was also up (p = 0.03).*
>
> *The uplift was strongest in the first two days. Recommendation: ship.*

In [ ]:
# Use this cell for any calculations that support your audit

---
### Before you hand in

- For every test, did you **check the split first**?
- Did you report the **interval**, not just the p-value?
- Did you state **how many metrics or segments** you looked at?
- Did you work out the **power**, and say so when a flat result was underpowered?
- Did you ever write "no effect" where you meant "we did not detect an effect"?
- For Part D, did you find all four problems? Both D tasks are marked.

Solutions will be released after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*